In [1]:
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: str | ForwardRef('os.PathLike[str]') | None = None, stream: IO[str] | None = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: str | None = 'utf-8') -> bool>

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings #embedding model -> gemini
from langchain_chroma import Chroma
from langchain_groq import ChatGroq #Output genration model -> groq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

In [2]:
#1.Load Data
loader = PyPDFLoader('../data/Ayush_resume.pdf')
docs = loader.load()
print(len(docs))

#2. Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=50 , chunk_overlap=10)
splitted_data = splitter.split_documents(docs)
print(len(splitted_data))

#3. Embedding Vector
# embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#4. Chroma db call that model and convert them into vectors and store insidei it
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
    persist_directory="./vector_db"
)


1
108


c:\Users\rajpu\Desktop\GEN-AI\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\rajpu\Desktop\GEN-AI\env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rajpu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to 

In [ ]:
#query phase
# query = "Full Stack developer"
# data = vector_store.similarity_search(query=query)

# #give similiarity search result as context + question/query to llm
# context=""
# for doc in data:
#     context += doc.page_content + "\n" #collect data from every page

# llm = ChatGroq(model='openai/gpt-oss-120b')
# res = llm.invoke(f"Provide me answer strictly based on context for my question, context:{context},question:{query}")
# print(res.content)

# Do this using chain: context-generate | prompt | llm | strparser

def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context += doc.page_content + "\n"

    return{
        "context":context,
        "question":query
    }

prompt = PromptTemplate.from_template("""Yor are helpful assistence and provide answer based on 
        context for user question and if you dont know the answer then say 'I dont know' in rude way.
        Context:{context}
        Question:{question}""")

llm = ChatGroq(model='openai/gpt-oss-120b')

rag_chain = get_context | prompt |llm #get_contex jo return krega wo prompt ka input hoga and prompt jo return krega wo llm ka input hoga

# rag_chain = rag_chain.invoke("Who is Ayush Rajput?")
# rag_chain = rag_chain.invoke("Who is Salman Khan?")
rag_chain = rag_chain.invoke("Give me Gen Ai skills?")

print(rag_chain.content)


Here are the Generative AI skills mentioned in the provided context:

- **Retrieval‑Augmented Generation (RAG)**
- **LangChain** (framework for building LLM‑powered applications)
- **Pinecone** (vector database for similarity search and embeddings)
- **Gemini** (Google’s family of generative‑AI models)
- **BabyCode** (presumably a tool or platform related to code generation)

These are the specific Gen AI technologies and frameworks referenced. If you need more detail on any of them, just let me know!
